## Batch mask prediction

Loops through every image in `input_dir`, runs the UNet and SAM segmentation pipeline on each one, and saves two training masks per image into `output_dir`:
- `<filename>_unet_mask.png` — argmax of UNet softmax output (0/1/2)
- `<filename>_sam_mask.png`  — SAM segmentation mask (0/1/2)
- `<filename>_image.png`     — source image copy (needed by `patchify_training_data`)

Pixel values: **0** = background, **1** = grain interior, **2** = grain boundary.

In [7]:
import os
import keras
import numpy as np
import segment_anything
import segmenteverygrain as seg
import segmenteverygrain.interactions as si
from pathlib import Path
from tqdm import tqdm

%matplotlib inline

In [8]:
# ── Paths ─────────────────────────────────────────────────────────────────────
input_dir  = './Images/cropped_alumina_highquality/'  # folder of images to process
output_dir = './prediction_outputs/'                  # where masks will be saved

# File extensions to process
extensions = {'.tif', '.tiff', '.jpg', '.jpeg', '.png'}

# ── Segmentation parameters ───────────────────────────────────────────────────
dbs_max_dist = 60.0   # increase to generate fewer SAM prompts (faster)
min_area     = 300.0  # minimum grain area in pixels

In [9]:
# Load UNet model
unet = keras.saving.load_model(
    './models/seg_model.keras',
    custom_objects={'weighted_crossentropy': seg.weighted_crossentropy},
)

# Download SAM model if needed
if not os.path.exists('./models/sam_vit_h_4b8939.pth'):
    import urllib.request
    urllib.request.urlretrieve(
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
        './models/sam_vit_h_4b8939.pth',
    )

# Load SAM
sam = segment_anything.sam_model_registry['default'](checkpoint='./models/sam_vit_h_4b8939.pth')
predictor = segment_anything.SamPredictor(sam)

In [10]:
os.makedirs(output_dir, exist_ok=True)

image_files = sorted(
    p for p in Path(input_dir).iterdir()
    if p.suffix.lower() in extensions
)

print(f'Found {len(image_files)} image(s) in {input_dir}')

for img_path in tqdm(image_files, desc='Processing images'):
    print(f'\n── {img_path.name}')
    out_stem = os.path.join(output_dir, img_path.stem)

    # Load image and run UNet
    image = si.load_image(str(img_path))
    predictor.set_image(image)
    image_pred = seg.predict_image(image, unet, I=256)

    # Generate SAM point prompts from UNet output
    labels, coords = seg.label_grains(image, image_pred, dbs_max_dist=dbs_max_dist)

    # Run SAM segmentation
    all_grains, labels, mask_all, grain_data, fig, ax = seg.sam_segmentation(
        sam, image, image_pred, coords, labels,
        min_area=min_area,
        plot_image=False,
        remove_edge_grains=True,
        remove_large_objects=False,
    )

    # Save training masks
    seg.save_training_masks(image, image_pred, mask_all, out_stem)
    print(f'   Saved masks to {out_stem}_[image|unet_mask|sam_mask].png')

print('\nDone.')

Found 91 image(s) in ./Images/cropped_alumina_highquality/


Processing images:   0%|          | 0/91 [00:00<?, ?it/s]


── cropped_prac7_etched_011.tif
segmenting image tiles...


100%|██████████| 5/5 [00:00<00:00,  7.62it/s]


creating masks using SAM...


...

finding best polygons...


Processing images:  11%|█         | 10/91 [03:29<29:17, 21.69s/it]

creating labeled image...
   Saved masks to ./prediction_outputs/cropped_prac7_etched_020_[image|unet_mask|sam_mask].png

── cropped_prac7_etched_021.tif
segmenting image tiles...


Processing images:  11%|█         | 10/91 [03:43<30:06, 22.31s/it]


KeyboardInterrupt: 